In [7]:
import pandas as pd
import matplotlib.pyplot as plt
#import seaborn as sns
import numpy as np
#from scipy import stats
#from statsmodels.genmod.cov_struct import Stationary
#from statsmodels.tsa.tsatools import detrend
#from pandas import Timedelta
#from pandas import TimedeltaIndex
#from noaa_coops import Station, get_stations_from_bbox
import tropycal.tracks as tracks
#import datetime as dt
#import copernicusmarine
#import intake
#import s3fs
#import json
from paratc.tc_models import Holland1980 as h80
import xarray as xr
from matplotlib import gridspec
from cmocean import cm as cmo
import netCDF4 as nc
from netCDF4 import Dataset, num2date
import dask.array as da
import os
import moviepy.video.io.ImageSequenceClip
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.io.img_tiles import OSM  # Ensure this is correctly imported



In [2]:
# Define Gulf of Mexico bounding box

lon_min = -98
lon_max = -81
lat_min = 18
lat_max = 30

gulf_bounds = {
 "min_lat": lat_min,
 "max_lat": lat_max,
 "min_lon": lon_min,
 "max_lon": lon_max
}

# Define the latitude and longitude bounds
lon_coords = [lon_min, lon_max]  # Minimum and maximum longitude for the Gulf of Mexico
lat_coords = [lat_min, lat_max]  # Minimum and maximum latitude for the Gulf of Mexico

In [3]:
# Filter storms passing through the Gulf of Mexico
def is_in_gulf(lat, lon):
 """
 Check if a single latitude/longitude point is in the Gulf of Mexico.
 """
 return (gulf_bounds['min_lat'] <= lat <= gulf_bounds['max_lat']) and \
  (gulf_bounds['min_lon'] <= lon <= gulf_bounds['max_lon'])


def storm_passed_through_gulf(storm_df):
 """
 Check if any track point of an individual storm falls within the Gulf of Mexico.
 """
 for _, row in storm_df.iterrows():
  if is_in_gulf(row['latitude'], row['longitude']):
   return True  # At least one point in Gulf
 return False

In [4]:
# Select Hurricane Season from which the Gulf of Mexico Hurricanes will be selected
season_year = 2017
# Get all the dataset from the Atlantic Basin
basin = tracks.TrackDataset(basin='north_atlantic')
season = basin.get_season(season_year)

df = season.to_dataframe()
# List to store storms that passed through the Gulf of Mexico
gulf_storms = []

# Iterate through all storms in the season
for storm_number in df.index:
 # Get storm object
 storm = basin.get_storm((df.name[storm_number], season_year))

 # Check all track points of the storm
 passed_through_gulf = False
 for lat, lon in zip(storm.dict['lat'], storm.dict['lon']):  # Extract track points
  if is_in_gulf(lat, lon):  # Check if the point is in the Gulf
   passed_through_gulf = True
   break  # Stop checking once a point is within the Gulf

 # If storm passed through the Gulf, add it to the list
 if passed_through_gulf:
  gulf_storms.append(storm)

# Output the storms that passed through the Gulf of Mexico
print("Storms that passed through the Gulf of Mexico in 2017:")
for storm in gulf_storms:
 print(storm.name)
 # OPTIONAL: Plot the storm's track
 #storm.plot()

--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (1.17 seconds)
Storms that passed through the Gulf of Mexico in 2017:
CINDY
EMILY
FRANKLIN
HARVEY
IRMA
KATIA
NATE
PHILIPPE


In [21]:
storm = gulf_storms[0]

# Convert storm data to DataFrame
track_df = pd.DataFrame({
    "time": storm.dict['time'],
    "lat": storm.dict['lat'],
    "lon": storm.dict['lon'],
    "vmax": storm.dict['vmax'],
    "pcen": storm.dict['mslp'],  # Central pressure
    "rmax": storm.dict.get('rmax', [30] * len(storm.dict['lat']))  # Default if missing
})
# Environmental pressure assumption
track_df["penv"] = 1010.0  # Default environmental pressure
track_df["pdelta"] = track_df["penv"] - track_df["pcen"]  # Pressure difference

# Define grid for wind field calculations
grid_lon, grid_lat = np.meshgrid(np.linspace(-100, -80, 50), np.linspace(20, 40, 50))

# Create ParaTC Holland1980 storm model
storm_model = h80(track_df, grid_lon, grid_lat, B_model='vickery00', interp_timestep=1)
nt = storm_model.data['time'].size

# Apply transformations
storm_model.scale_winds(0.91)
storm_model.apply_inflow_angle(inflow_model='nws')
storm_model.add_background_winds(bg_alpha=0.55, bg_beta=20)
storm_model.make_wind_stress(cd_model='garratt77', cd_max=3e-3)


C:\Users\ebear\AppData\Local\anaconda3\Lib\site-packages\paratc\track_tools.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  new_time = pd.date_range( time_0[0], time_0[-1], freq=f'{new_timestep}H')
C:\Users\ebear\AppData\Local\anaconda3\Lib\site-packages\paratc\track_tools.py:62: RuntimeWarning: invalid value encountered in divide
  utrans = trans_speed * utrans / vec_norm
C:\Users\ebear\AppData\Local\anaconda3\Lib\site-packages\paratc\track_tools.py:63: RuntimeWarning: invalid value encountered in divide
  vtrans = trans_speed * vtrans / vec_norm


In [19]:
from moviepy import ImageSequenceClip

# Create a directory to store plot images
output_presdir_map = "output_pres_map"
output_winddir_map = "output_wind_map"
os.makedirs(output_presdir_map, exist_ok=True)
os.makedirs(output_winddir_map, exist_ok=True)

# Generate and save plots as images
image_files_wind_map = []
image_files_pres_map = []

for tt in range(0, nt):
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.PlateCarree()})

    windspeed = storm_model.data['windspeed'][tt].values
    pressure = storm_model.data['pressure'][tt].values
    lon = storm_model.data['lon'].values
    lat = storm_model.data['lat'].values

    im = ax.contourf(lon, lat, windspeed, transform=ccrs.PlateCarree(), cmap='coolwarm', alpha=0.6)
    plt.colorbar(im, ax=ax, orientation='vertical', label='Wind Speed (m/s)')
    time_value = storm_model.data['time'][tt].values
    if isinstance(time_value, np.datetime64):
        time_value = pd.Timestamp(time_value)  # Pandas Timestamp works with strftime
    ax.set_title(f"Wind Speed - Time: {time_value.strftime('%Y-%m-%d %H:%M')}")
    osm_tiles = OSM()
    ax.add_image(osm_tiles, 8)
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER

    #plt.show()  # Display the current plot
    windspeed_image_path = os.path.join(output_winddir_map, f"windspeed_{tt:03d}.png")
    plt.savefig(windspeed_image_path)
    image_files_wind_map.append(windspeed_image_path)  # Add windspeed image to list
    plt.close()

# Create a video from the saved images
fps = 4  # Frames per second
clip_wind = ImageSequenceClip(image_files_wind_map, fps=fps)
video_file_wind = f"{storm.name}_storm_animation_wind_map.mp4"
clip_wind.write_videofile(video_file_wind, codec="libx264")
print(f"Video saved as {video_file_wind}")


MoviePy - Building video EMILY_storm_animation_wind_map.mp4.
MoviePy - Writing video EMILY_storm_animation_wind_map.mp4



MoviePy - Done !
MoviePy - video ready EMILY_storm_animation_wind_map.mp4
MoviePy - Building video EMILY_storm_animation_pres_map.mp4.
MoviePy - Writing video EMILY_storm_animation_pres_map.mp4



MoviePy - Done !
MoviePy - video ready EMILY_storm_animation_pres_map.mp4
Video saved as EMILY_storm_animation_wind_map.mp4
Video saved as EMILY_storm_animation_pres_map.mp4


In [22]:
from moviepy import ImageSequenceClip

# Create a directory to store plot images
output_presdir_map = "output_pres_map"
os.makedirs(output_presdir_map, exist_ok=True)

# Generate and save plots as images
image_files_pres_map = []

for tt in range(0, nt):
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.PlateCarree()})

    pressure = storm_model.data['pressure'][tt].values
    lon = storm_model.data['lon'].values
    lat = storm_model.data['lat'].values

    im = ax.contourf(lon, lat, pressure, transform=ccrs.PlateCarree(), cmap='coolwarm', alpha=0.6)
    plt.colorbar(im, ax=ax, orientation='vertical', label='Pressure (hPa)')
    time_value = storm_model.data['time'][tt].values
    if isinstance(time_value, np.datetime64):
        time_value = pd.Timestamp(time_value)  # Pandas Timestamp works with strftime
    ax.set_title(f"Wind Speed - Time: {time_value.strftime('%Y-%m-%d %H:%M')}")
    osm_tiles = OSM()
    ax.add_image(osm_tiles, 8)
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    plt.title(f"Pressure Field for Hurricane {storm.name} at {time_value.strftime('%Y-%m-%d %H:%M')}")
    pressure_image_path = os.path.join(output_presdir_map, f"pressure_{tt:03d}.png")
    plt.savefig(pressure_image_path)
    image_files_pres_map.append(pressure_image_path)  # Add pressure image to list
    plt.close()


fps = 4  # Frames per second
clip_pres = ImageSequenceClip(image_files_pres_map, fps=fps)
video_file_pres = f"{storm.name}_storm_animation_pres_map.mp4"
clip_pres.write_videofile(video_file_pres, codec="libx264")
print(f"Video saved as {video_file_pres}")

MoviePy - Building video CINDY_storm_animation_pres_map.mp4.
MoviePy - Writing video CINDY_storm_animation_pres_map.mp4



MoviePy - Done !
MoviePy - video ready CINDY_storm_animation_pres_map.mp4
Video saved as CINDY_storm_animation_pres_map.mp4


In [28]:
# Create a directory to store plot images
output_presdir = "output_pres"
output_winddir = "output_wind"
os.makedirs(output_presdir, exist_ok=True)
os.makedirs(output_winddir, exist_ok=True)

# Generate and save plots as images
image_files_wind = []
image_files_pres = []
for tt in range(0, nt):
    # Plot windspeed field
    storm_model.plot(tt, field='windspeed')

    # Ensure .values is used to extract datetime-like object:
    time_value = storm_model.data['time'][tt].values
    # Convert to datetime if needed (e.g., for numpy.datetime64):
    if isinstance(time_value, np.datetime64):
        time_value = pd.Timestamp(time_value)  # Pandas Timestamp works with strftime

    plt.title(f"Wind Field for Hurricane {storm.name} at {time_value.strftime('%Y-%m-%d %H:%M')}")
    windspeed_image_path = os.path.join(output_winddir, f"windspeed_{tt:03d}.png")
    plt.savefig(windspeed_image_path)
    image_files_wind.append(windspeed_image_path)  # Add windspeed image to list
    plt.close()

    # Plot pressure field
    storm_model.plot(tt, field='pressure')
    plt.title(f"Pressure Field for Hurricane {storm.name} at {time_value.strftime('%Y-%m-%d %H:%M')}")
    pressure_image_path = os.path.join(output_presdir, f"pressure_{tt:03d}.png")
    plt.savefig(pressure_image_path)
    image_files_pres.append(pressure_image_path)  # Add pressure image to list
    plt.close()

# Create a video from the saved images
fps = 4  # Frames per second
clip_wind = ImageSequenceClip(image_files_wind, fps=fps)
clip_pres = ImageSequenceClip(image_files_pres, fps=fps)
video_file_wind = f"{storm.name}_storm_animation_wind.mp4"
video_file_pres = f"{storm.name}_storm_animation_pres.mp4"
clip_wind.write_videofile(video_file_wind, codec="libx264")
clip_pres.write_videofile(video_file_pres, codec="libx264")
print(f"Video saved as {video_file_wind}")
print(f"Video saved as {video_file_pres}")

C:\Users\ebear\AppData\Local\anaconda3\Lib\site-packages\matplotlib\quiver.py:649: RuntimeWarning: divide by zero encountered in scalar divide
  length = a * (widthu_per_lenu / (self.scale * self.width))
C:\Users\ebear\AppData\Local\anaconda3\Lib\site-packages\matplotlib\quiver.py:649: RuntimeWarning: invalid value encountered in multiply
  length = a * (widthu_per_lenu / (self.scale * self.width))
C:\Users\ebear\AppData\Local\anaconda3\Lib\site-packages\matplotlib\quiver.py:649: RuntimeWarning: divide by zero encountered in scalar divide
  length = a * (widthu_per_lenu / (self.scale * self.width))
C:\Users\ebear\AppData\Local\anaconda3\Lib\site-packages\matplotlib\quiver.py:649: RuntimeWarning: invalid value encountered in multiply
  length = a * (widthu_per_lenu / (self.scale * self.width))


MoviePy - Building video CINDY_storm_animation_wind.mp4.
MoviePy - Writing video CINDY_storm_animation_wind.mp4



MoviePy - Done !
MoviePy - video ready CINDY_storm_animation_wind.mp4
MoviePy - Building video CINDY_storm_animation_pres.mp4.
MoviePy - Writing video CINDY_storm_animation_pres.mp4



MoviePy - Done !
MoviePy - video ready CINDY_storm_animation_pres.mp4
Video saved as CINDY_storm_animation_wind.mp4
Video saved as CINDY_storm_animation_pres.mp4


[[1010. 1010. 1010. ... 1010. 1010. 1010.]
 [1010. 1010. 1010. ... 1010. 1010. 1010.]
 [1010. 1010. 1010. ... 1010. 1010. 1010.]
 ...
 [1010. 1010. 1010. ... 1010. 1010. 1010.]
 [1010. 1010. 1010. ... 1010. 1010. 1010.]
 [1010. 1010. 1010. ... 1010. 1010. 1010.]]
